In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, make_scorer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, train_test_split

In [ ]:
# Chọn uncomment 1 trong các file sau để chọn training set tương ứng cho từng thực nghiệm

df_train = pd.read_csv(r"/root/train.csv")                              # w/o augmentation
# df_train = pd.read_csv(r"/root/train_eda.csv")                          # EDA augmentation
# df_train = pd.read_csv(r"/root/train_aeda.csv")                         # AEDA augmentation
# df_train = pd.read_csv(r"/root/train_ease.csv")                         # EASE augmentation
# df_train = pd.read_csv(r"/root/train-llm-aug-gpt-5-2025-08-07.csv")     # LLM augmentation (gpt-5-2025-08-07)

df_val = pd.read_csv(r"/root/val.csv")
df_test = pd.read_csv(r"/root/test.csv")

In [ ]:
# mapping categorical label sang numerical label, tăng dần theo mức độ nghiêm trọng của bug, ML models quan tâm đến ordering nên cần set tăng dần. Bắt đầu từ 0 để consist với modern NLP methods sau này.

mapping_list = {
    'Trivial': 0,
    'Minor': 1,
    'Major': 2,
    'Critical': 3,
    'Blocker': 4
    }

# ===== UNDER_SAMPLING WITH REAL TRAINING DATA =====
# def process_training_set_with_balance_class_distribution(dataframe, target_column_name="priority"):
#     min_sample = dataframe[target_column_name].value_counts().min()
    
#     df_filtered = dataframe.groupby(target_column_name, group_keys=False).apply(
#         lambda x: x.sample(n=min_sample, random_state=42)
#     ).reset_index(drop=True)
    
#     return df_filtered

# df_train = process_training_set_with_balance_class_distribution(dataframe=df_train)

# ===== OVER_SAMPLING WITH REAL TRAINING DATA =====
# def process_training_set_with_oversampling(dataframe, target_column_name="priority"):
#     max_sample = dataframe[target_column_name].value_counts().max()
    
#     df_oversampled = dataframe.groupby(target_column_name, group_keys=False).apply(
#         lambda x: x.sample(n=max_sample, replace=True, random_state=42)
#     ).reset_index(drop=True)
    
#     return df_oversampled

# df_train = process_training_set_with_oversampling(dataframe=df_train)

y_train = df_train['priority'].map(mapping_list)
y_val = df_val['priority'].map(mapping_list)
y_test = df_test['priority'].map(mapping_list)

In [ ]:
all_text = ' '.join(df_train["text"].astype(str))
vocab = set(all_text.lower().split())
print(f"Vocab size (simple): {len(vocab)}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# để vocab cho TFIDF là 5000 để cân bằng giữa speed và performance, spare matrix với số chiều lớn huấn luyện lâu, tinh chỉnh siêu tham số cũng lâu, dễ vướng vào 'curse of dimensionality', lọc stopword để tránh vocab chữa những từ không mang nhiều ý nghĩa, và tăng hiệu quả embedding khi nhiều từ vựng mang ý nghĩa xuất hiện hơn.
tfidf = TfidfVectorizer(max_features=5000,
                        stop_words='english')

X_train = tfidf.fit_transform(df_train["text"])
X_val = tfidf.transform(df_val["text"])
X_test = tfidf.transform(df_test["text"])

# Training baselines

In [ ]:
def train_and_evaluate(model,
                       X_train=X_train,
                       y_train=y_train,
                       X_test=X_test,
                       y_test=y_test):

    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    
    print(f"Accuracy: {round(accuracy_score(y_true=y_test, y_pred=y_test_pred)*100, 2)}")
    print(f"Micro Precsion: {round(precision_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='micro')*100, 2)}\tMacro Precsion: {round(precision_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='macro')*100, 2)}")
    print(f"Micro Recall: {round(recall_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='micro')*100, 2)}\tMacro Recall: {round(recall_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='macro')*100, 2)}")
    print(f"Micro F1: {round(f1_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='micro')*100, 2)}\t\tMacro F1: {round(f1_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='macro')*100, 2)}")
    print("="*60)
    print(f"Confusion Matrix:\n{confusion_matrix(y_true=y_test, y_pred=y_test_pred)}")
    print("="*60)
    print(f"Classification Report:\n{classification_report(y_true=y_test, y_pred=y_test_pred, zero_division=0)}")

In [ ]:
def evaluate(model,
             X_test=X_test,
             y_test=y_test):

    y_test_pred = model.predict(X_test)

    print(f"Accuracy: {round(accuracy_score(y_true=y_test, y_pred=y_test_pred)*100, 2)}")
    print(f"Micro Precsion: {round(precision_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='micro')*100, 2)}\tMacro Precsion: {round(precision_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='macro')*100, 2)}")
    print(f"Micro Recall: {round(recall_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='micro')*100, 2)}\tMacro Recall: {round(recall_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='macro')*100, 2)}")
    print(f"Micro F1: {round(f1_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='micro')*100, 2)}\t\tMacro F1: {round(f1_score(y_true=y_test, y_pred=y_test_pred, zero_division=0, average='macro')*100, 2)}")
    print("="*60)
    print(f"Confusion Matrix:\n{confusion_matrix(y_true=y_test, y_pred=y_test_pred)}")
    print("="*60)
    print(f"Classification Report:\n{classification_report(y_true=y_test, y_pred=y_test_pred, zero_division=0)}")

In [ ]:
train_and_evaluate(model=LogisticRegression(random_state=42))

In [ ]:
train_and_evaluate(model=XGBClassifier(random_state=42))

# HYPER-PARAMENTER TUNING VIA GRID-SEARCH + 5-FOLD CROSS-VALIDATION

In [ ]:
def grid_search(model, param_grid, X_train, y_train, X_val, y_val, cv=5, n_jobs=-1, verbose=2):    
    print("="*80)
    print("GRID SEARCH")
    print("="*80)
    
    print("Parameter Grid:")
    for param, values in param_grid.items():
        print(f"   {param}: {values}")
    print()
    
    # Scorer: Macro F1
    scorer = make_scorer(f1_score, average='macro')
    
    print("Grid Searching...")
    print()
    
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scorer,
        cv=cv,
        n_jobs=n_jobs,
        verbose=verbose,
        return_train_score=True
    )
    
    grid_search.fit(X_train, y_train)
    
    print("\n" + "="*80)
    print("GRID SEARCH COMPLETED")
    print("="*80)
    
    print("\nBEST PARAMETERS:")
    for param, value in grid_search.best_params_.items():
        print(f"   {param}: {value}")
    
    print(f"\nBest CV Macro F1 Score: {grid_search.best_score_:.4f}")
    
    best_model = grid_search.best_estimator_
    y_val_pred = best_model.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    print(f"Validation Macro F1 Score: {val_f1:.4f}")
    
    print("\n" + "="*80)
    print("VALIDATION SET PERFORMANCE")
    print("="*80)
    
    # Classification report
    target_names = ['Trivial', 'Minor', 'Major', 'Critical', 'Blocker']
    print(classification_report(y_val, y_val_pred, target_names=target_names, digits=4))
    
    return best_model

In [ ]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['saga'],
    'class_weight': ['balanced', None],
    'max_iter': [1000]
    }

best_model = grid_search(model=LogisticRegression(random_state=42),
                         param_grid=param_grid,
                         X_train=X_train,
                         y_train=y_train,
                         X_val=X_val,
                         y_val=y_val,
                         cv=5,
                         n_jobs=-1,
                         verbose=2)

evaluate(model=best_model)

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9, 11],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0, 0.01, 0.1, 1]
}

best_model = grid_search(model=XGBClassifier(random_state=42),
                         param_grid=param_grid,
                         X_train=X_train,
                         y_train=y_train,
                         X_val=X_val,
                         y_val=y_val,
                         cv=5,
                         n_jobs=-1,
                         verbose=2)

evaluate(model=best_model)

# MAI BỔ SUNG CODE HYPER-PARAMETER TUNING CHO ML-BASED MODELS, CODE TRAINING DL-BASED MODELS, TRANSFORMER-BASED MODELS